## Q3 — What is the impact? "Which ecosystems and species are most affected by ocean plastic?"

In [1]:
from itertools import combinations
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import mannwhitneyu, kruskal, chi2_contingency, randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import BallTree
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import yaml


In [2]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

In [3]:
# Load datasets
species = pd.read_parquet(config["output_data"]["file9"])
microplastics = pd.read_parquet(config["output_data"]["file3"])

# Basic shape
print("=== SPECIES ===")
print(species.shape)
print(species[["latitude", "longitude", "has_ingestion"]].describe())
print("\nMissing values:")
print(species[["latitude", "longitude", "has_ingestion"]].isnull().sum())

print("\n=== MICROPLASTICS ===")
print(microplastics.shape)
print(microplastics[["lat", "lng", "concentration_class"]].describe())
print("\nconcentration_class unique values:")
print(microplastics["concentration_class"].value_counts())
print("\nMissing values:")
print(microplastics[["lat", "lng", "concentration_class"]].isnull().sum())

=== SPECIES ===
(10412, 47)
           latitude     longitude  has_ingestion
count  10205.000000  10205.000000   10412.000000
mean      20.524977    -29.676304       0.181329
std       28.386640     77.937760       0.385309
min      -53.000000   -178.000000       0.000000
25%       25.770000    -81.738710       0.000000
50%       27.673040    -80.238262       0.000000
75%       29.028847      8.500000       0.000000
max       78.380000    180.000000       1.000000

Missing values:
latitude         207
longitude        207
has_ingestion      0
dtype: int64

=== MICROPLASTICS ===
(22530, 14)
                lat           lng
count  22530.000000  22530.000000
mean      26.196529    -65.811942
std       18.347551     68.993855
min      -71.699040   -179.994200
25%       19.962500    -97.162100
50%       28.067050    -79.050800
75%       33.870000    -59.450000
max       89.761400    179.854333

concentration_class unique values:
concentration_class
Medium       10664
Very Low      5950
Hig

In [4]:
# Step 2 — Clean both datasets

# Drop species rows with missing coordinates
species_clean = species.dropna(subset=["latitude", "longitude"]).copy()
print(f"Species: {len(species)} → {len(species_clean)} rows after dropping missing coords")
print(f"Dropped: {len(species) - len(species_clean)} rows ({(len(species) - len(species_clean)) / len(species):.1%})")

# Standardise concentration_class — strip whitespace, fix capitalisation
microplastics["concentration_class"] = microplastics["concentration_class"].str.strip()

# Define the correct order low → high
class_order = ["Very Low", "Low", "Medium", "High", "Very High"]

# Make it an ordered categorical so pandas respects the order in charts and groupbys
microplastics["concentration_class"] = pd.Categorical(
    microplastics["concentration_class"],
    categories=class_order,
    ordered=True
)

# Confirm
print(f"\nMicroplastics concentration_class after cleaning:")
print(microplastics["concentration_class"].value_counts().sort_index())
print(f"\nAny unexpected values: {microplastics['concentration_class'].isnull().sum()}")

Species: 10412 → 10205 rows after dropping missing coords
Dropped: 207 rows (2.0%)

Microplastics concentration_class after cleaning:
concentration_class
Very Low      5950
Low           2516
Medium       10664
High          2642
Very High      758
Name: count, dtype: int64

Any unexpected values: 0


In [5]:
# Step 3 — Spatial join

RADIUS_KM = 50
EARTH_RADIUS_KM = 6371.0

# Convert coordinates to radians — BallTree requires this for haversine
mp_coords_rad = np.radians(microplastics[["lat", "lng"]].values)
species_coords_rad = np.radians(species_clean[["latitude", "longitude"]].values)

# Build the tree on microplastic sample points
tree = BallTree(mp_coords_rad, metric="haversine")

# For each animal, find the single nearest microplastic sample
# distances come back in radians — we convert to km after
distances_rad, indices = tree.query(species_coords_rad, k=1)
distances_km = distances_rad[:, 0] * EARTH_RADIUS_KM

# Tag each animal with its distance and whether it falls within 50 km
species_clean["dist_to_nearest_mp_km"] = distances_km
species_clean["within_radius"] = distances_km <= RADIUS_KM

# Pull the concentration_class from the nearest matched microplastic sample
nearest_idx = indices[:, 0]
species_clean["concentration_class"] = microplastics["concentration_class"].iloc[nearest_idx].values

# Mask animals outside the radius — they get NaN, excluded from analysis
species_clean.loc[~species_clean["within_radius"], "concentration_class"] = np.nan

# Quick summary
n_matched = species_clean["within_radius"].sum()
n_total = len(species_clean)
print(f"Matched within {RADIUS_KM} km : {n_matched:,} / {n_total:,} ({n_matched/n_total:.1%})")
print(f"\nMatched animals by concentration class:")
print(species_clean["concentration_class"].value_counts().sort_index())

Matched within 50 km : 6,652 / 10,205 (65.2%)

Matched animals by concentration class:
concentration_class
Very Low     2707
Low           588
Medium       2954
High          230
Very High     173
Name: count, dtype: int64


In [6]:
# Step 4 — Ingestion rate by concentration class

matched = species_clean[species_clean["within_radius"]].copy()

ingestion_by_class = (
    matched.groupby("concentration_class", observed=True)["has_ingestion"]
    .agg(
        n_animals="count",
        n_with_ingestion="sum"
    )
    .assign(ingestion_rate=lambda d: d["n_with_ingestion"] / d["n_animals"] * 100)
    .reset_index()
)

print(ingestion_by_class.to_string(index=False))

concentration_class  n_animals  n_with_ingestion  ingestion_rate
           Very Low       2707               347       12.818618
                Low        588                99       16.836735
             Medium       2954               521       17.637102
               High        230                82       35.652174
          Very High        173                24       13.872832


It's not monotonic — it rises nicely from Very Low through High, but then Very High drops back down to almost the same level as Very Low. That's unexpected and actually makes for a more interesting story than a clean trend would.
Before we run the statistics, let's think about what could explain the Very High drop:

- Only 173 animals in that class — the smallest group by far, so it could be noise
- Very High concentration zones might be offshore/remote areas where certain species (that happen to ingest less) are more commonly studied
- Or it could be a genuine ecological effect worth investigating

The statistics in step 5 will tell whether any of these differences are actually significant or just sampling noise. 


Why Kruskal-Wallis?
It's the non-parametric equivalent of ANOVA — it makes no assumptions about the distribution of your data. Instead of comparing means, it works on ranks — it lines up all 6,652 observations from lowest to highest and asks whether the ranks are distributed evenly across the five groups.

It's the standard choice when you have:
- Binary or skewed data ✓
- Multiple groups (more than 2) ✓
- Unequal group sizes ✓

In [7]:
# Step 5 — Kruskal-Wallis test
# Non-parametric — appropriate here because has_ingestion is binary (0/1)
# and our groups are very different sizes
groups = [
    matched[matched["concentration_class"] == cls]["has_ingestion"].values
    for cls in ["Very Low", "Low", "Medium", "High", "Very High"]
]

stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H = {stat:.3f}")
print(f"p-value           = {p:.4f}")

if p < 0.05:
    print("\n→ Significant difference exists across classes (p < 0.05)")
    print("  We'll now run pairwise tests to find which classes differ")
else:
    print("\n→ No significant difference across classes (p ≥ 0.05)")

Kruskal-Wallis H = 92.549
p-value           = 0.0000

→ Significant difference exists across classes (p < 0.05)
  We'll now run pairwise tests to find which classes differ


H = 92.5 with p ≈ 0.0000 means the difference in ingestion rates across the five classes is extremely unlikely to be random. 
But Kruskal-Wallis only tells "at least one group differs" — it doesn't say which ones.

Step 6 — pairwise Mann-Whitney tests.
Why Mann-Whitney?
It's the non-parametric equivalent of a t-test — it compares two groups at a time using the same rank-based logic as Kruskal-Wallis. 
We run it for every possible pair of classes.

Why Bonferroni correction?
With 5 classes we have 10 possible pairs. If we test each pair at p < 0.05, the chance of getting at least one false positive by pure luck is much higher than 5%. Bonferroni corrects for this by dividing the threshold by the number of tests — so we require p < 0.005 (0.05 / 10) instead.

In [8]:
class_order = ["Very Low", "Low", "Medium", "High", "Very High"]
pairs = list(combinations(class_order, 2))
n_comparisons = len(pairs)
bonferroni_threshold = 0.05 / n_comparisons

print(f"Number of pairs      : {n_comparisons}")
print(f"Bonferroni threshold : {bonferroni_threshold:.4f}\n")

results = []
for cls_a, cls_b in pairs:
    a = matched[matched["concentration_class"] == cls_a]["has_ingestion"].values
    b = matched[matched["concentration_class"] == cls_b]["has_ingestion"].values
    stat, p_raw = mannwhitneyu(a, b, alternative="two-sided")
    p_adj = min(p_raw * n_comparisons, 1.0)
    significant = p_adj < 0.05
    results.append({
        "pair": f"{cls_a} vs {cls_b}",
        "p_adjusted": round(p_adj, 4),
        "significant": "✓" if significant else "—"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Number of pairs      : 10
Bonferroni threshold : 0.0050

                 pair  p_adjusted significant
      Very Low vs Low      0.0985           —
   Very Low vs Medium      0.0000           ✓
     Very Low vs High      0.0000           ✓
Very Low vs Very High      1.0000           —
        Low vs Medium      1.0000           —
          Low vs High      0.0000           ✓
     Low vs Very High      1.0000           —
       Medium vs High      0.0000           ✓
  Medium vs Very High      1.0000           —
    High vs Very High      0.0000           ✓


High concentration is the real outlier — it's significantly different from every single other class. 
That 35.6% ingestion rate we saw earlier is genuinely unusual, not sampling noise.
Very High is the mystery — it's not significantly different from Very Low, Low, or Medium, only from High. 
Combined with its small sample size (173 animals) this strongly suggests the Very High drop is a sampling artefact rather than a real ecological effect. 
Too few observations to be reliable.
The lower classes (Very Low, Low, Medium) are largely indistinguishable from each other — the ingestion rates of 12.8%, 16.8% and 17.6% are too close to tell apart statistically.

Animals observed in High microplastic concentration zones show significantly elevated ingestion rates (35.6%) compared to all other concentration classes. 
The relationship is not cleanly monotonic — the Very High class behaves anomalously, likely due to small sample size.

In [9]:
class_order = ["Very Low", "Low", "Medium", "High", "Very High"]
colors = {
    "Very Low": "#5DCAA5",
    "Low":      "#5DCAA5",
    "Medium":   "#5DCAA5",
    "High":     "#E24B4A",
    "Very High": "#5DCAA5",
}

fig = go.Figure()

for i, row in ingestion_by_class.iterrows():
    cls = row["concentration_class"]
    fig.add_trace(go.Bar(
        x=[str(cls)],
        y=[row["ingestion_rate"]],
        marker_color=colors[str(cls)],
        text=f"{row['ingestion_rate']:.1f}%<br>n={int(row['n_animals'])}",
        textposition="outside",
        width=0.5,
        showlegend=False,
    ))

# Significance brackets — High vs each other class
bracket_pairs = [
    ("Very Low", "High", "p<0.001"),
    ("Low",      "High", "p<0.001"),
    ("Medium",   "High", "p<0.001"),
]

# Y positions for brackets — stacked above the tallest bar
bracket_y = [40, 44, 48]

for (cls_a, cls_b, label), y in zip(bracket_pairs, bracket_y):
    x_a = class_order.index(cls_a)
    x_b = class_order.index(cls_b)
    # Horizontal line
    fig.add_shape(type="line",
        x0=x_a, x1=x_b, y0=y, y1=y,
        line=dict(color="#444", width=1))
    # Left tick
    fig.add_shape(type="line",
        x0=x_a, x1=x_a, y0=y-0.5, y1=y,
        line=dict(color="#444", width=1))
    # Right tick
    fig.add_shape(type="line",
        x0=x_b, x1=x_b, y0=y-0.5, y1=y,
        line=dict(color="#444", width=1))
    # Label
    fig.add_annotation(
        x=(x_a + x_b) / 2, y=y + 0.5,
        text=label, showarrow=False,
        font=dict(size=10, color="#444")
    )

fig.update_layout(
    title=dict(
        text="H1 — Plastic ingestion rate by microplastic concentration class",
        font=dict(size=15)
    ),
    xaxis=dict(
        title="Microplastic concentration class",
        categoryorder="array",
        categoryarray=class_order,
    ),
    yaxis=dict(
        title="Animals with plastic ingestion (%)",
        range=[0, 55],
        ticksuffix="%",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=135, l=60, r=40),
    width=700,
    height=480,
)

fig.add_annotation(
    text="* Significance brackets show Bonferroni-corrected Mann-Whitney tests (α=0.05). High vs Very High also significant (p<0.001).",
    xref="paper", yref="paper",
    x=0, y=-0.12,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig.show()
fig.write_html("../figures/q3_h1_ingestion_by_concentration.html")

### H1 — Conclusion

**Hypothesis:** Marine species found in high microplastic concentration areas 
show higher plastic ingestion rates.

**Result: Partially confirmed**

A spatial join (BallTree haversine, 50 km radius) matched 6,652 of 10,205 
species observations (65.2%) to a microplastic sample point. Ingestion rates 
by concentration class:

| Class    | Animals | Ingestion rate |
|----------|---------|----------------|
| Very Low | 2,707   | 12.8%          |
| Low      | 588     | 16.8%          |
| Medium   | 2,954   | 17.6%          |
| High     | 230     | **35.6%**      |
| Very High| 173     | 13.9%          |

Kruskal-Wallis confirmed a significant difference across classes 
(H=92.5, p<0.001). Pairwise Mann-Whitney tests (Bonferroni corrected) 
show the **High class is significantly different from all other classes**.

The relationship is not monotonic — the Very High class drops back to 
baseline levels (13.9%), likely due to small sample size (n=173) rather 
than a genuine ecological effect.

**Limitation:** 34.8% of species observations had no microplastic sample 
within 50 km and were excluded from the analysis.

### H2: Are large animals disproportionately represented among the 1,019 ghost net entanglement cases?

In [10]:
print(species[["species", "group", "size_avg"]].groupby("group")["size_avg"].describe().round(1))
print("\nSome examples of large animals (is_large=1):")
print(species[species["is_large"]==1][["species", "group", "size_avg"]].drop_duplicates("species").sort_values("size_avg", ascending=False).head(20))

                count     mean      std     min      25%      50%      75%  \
group                                                                        
Albatross       357.0      4.0      1.5     1.0      2.8      3.9      3.9   
Baleen Whale      5.0  50439.4  25607.5  9072.0  54431.0  54431.0  54431.0   
Booby            13.0      1.3      0.0     1.3      1.3      1.3      1.3   
Cormorant         3.0      1.3      0.0     1.3      1.3      1.3      1.3   
Frigate Bird      4.0      1.4      0.0     1.4      1.4      1.4      1.4   
Fulmar           29.0      0.7      0.0     0.7      0.7      0.7      0.7   
Gannet            1.0      2.9      NaN     2.9      2.9      2.9      2.9   
Guillemot        30.0      0.4      0.0     0.4      0.4      0.4      0.4   
Gull             24.0      1.9      0.3     1.0      2.0      2.0      2.0   
Kittiwake        20.0      0.3      0.0     0.3      0.3      0.3      0.3   
Manatee        4950.0    453.6      0.0   453.6    453.6    453.

**Large animals in this dataset (is_large=1, size_avg ≥ 200) are primarily:**
- Whales (Baleen and Toothed) — including Sperm, Fin, Orca, Pilot whales
- Manatees
- Large turtles (e.g. Leatherback)
- Large Pinnipeds

**Note:** size_avg units appear inconsistent across groups (likely cm for 
most species, kg for whales). The is_large threshold should be interpreted 
as a relative size indicator rather than an absolute measurement.

In [11]:
print("=== ghost_net_entanglement ===")
print(species["ghost_net_entanglement"].value_counts(dropna=False))

print("\n=== is_large ===")
print(species["is_large"].value_counts(dropna=False))

print("\n=== is_entangled ===")
print(species["is_entangled"].value_counts(dropna=False))

print("\n=== fisheries / rope / line (first 5 rows) ===")
print(species[["fisheries", "rope", "line"]].head(10))
print("\nNon-zero counts:")
print((species[["fisheries", "rope", "line"]] > 0).sum())

=== ghost_net_entanglement ===
ghost_net_entanglement
0    9393
1    1019
Name: count, dtype: int64

=== is_large ===
is_large
1    5460
0    4952
Name: count, dtype: int64

=== is_entangled ===
is_entangled
0    9393
1    1019
Name: count, dtype: int64

=== fisheries / rope / line (first 5 rows) ===
   fisheries  rope  line
0        0.0     0   0.0
1        0.0     0   0.0
2        0.0     0   0.0
3        0.0     0   0.0
4        0.0     0   0.0
5        0.0     0   0.0
6        0.0     0   0.0
7        0.0     0   0.0
8        0.0     0   0.0
9        0.0     0   0.0

Non-zero counts:
fisheries    940
rope         157
line         749
dtype: int64


In [12]:
# Cross-tabulation of is_large vs ghost_net_entanglement
crosstab = pd.crosstab(
    species["is_large"],
    species["ghost_net_entanglement"],
    rownames=["is_large"],
    colnames=["ghost_net_entangled"],
    margins=True
)

print(crosstab)

# Also as percentages — what share of large vs small animals are entangled?
print("\nEntanglement rate by size:")
rate = pd.crosstab(
    species["is_large"],
    species["ghost_net_entanglement"],
    normalize="index"
) * 100
print(rate.round(2))

ghost_net_entangled     0     1    All
is_large                              
0                    4547   405   4952
1                    4846   614   5460
All                  9393  1019  10412

Entanglement rate by size:
ghost_net_entanglement      0      1
is_large                            
0                       91.82   8.18
1                       88.75  11.25


So large animals have an 11.25% entanglement rate versus 8.18% for small animals. That's a real difference — large animals are about 37% more likely to be ghost net entangled than small ones.
But before we get excited, let's think about what the statistics need to confirm:

Is this difference statistically significant, or could it be chance?
Is it practically meaningful, or just a small effect that happens to be significant because we have 10,000 rows?

In [13]:
# The 2x2 contingency table (without margins)
contingency = pd.crosstab(
    species["is_large"],
    species["ghost_net_entanglement"]
)

print("Contingency table:")
print(contingency)

# Chi-square test
chi2, p, dof, expected = chi2_contingency(contingency)
print(f"\nChi-square statistic : {chi2:.3f}")
print(f"p-value              : {p:.4f}")
print(f"Degrees of freedom   : {dof}")

# Odds ratio — how much more likely are large animals to be entangled?
a = contingency.loc[1, 1]  # large + entangled
b = contingency.loc[1, 0]  # large + not entangled
c = contingency.loc[0, 1]  # small + entangled
d = contingency.loc[0, 0]  # small + not entangled

odds_ratio = (a * d) / (b * c)
print(f"\nOdds ratio           : {odds_ratio:.3f}")
print("(>1 means large animals are more likely to be entangled)")

Contingency table:
ghost_net_entanglement     0    1
is_large                         
0                       4547  405
1                       4846  614

Chi-square statistic : 27.319
p-value              : 0.0000
Degrees of freedom   : 1

Odds ratio           : 1.423
(>1 means large animals are more likely to be entangled)


Chi-square: p < 0.001 — the difference is highly significant, definitely not chance.
Odds ratio: 1.423 — large animals are 42% more likely to be ghost net entangled than small animals.

So the hypothesis is confirmed — but with an important nuance worth noting. 
The effect exists and is statistically robust, but 42% more likely is a moderate effect, not a dramatic one. 
Small animals still get entangled too (8.18%) — ghost nets don't exclusively target large animals, they just disproportionately affect them.
This is actually a more honest and interesting finding than "large animals always get caught" — it suggests ghost nets are indiscriminate but size does matter.

In [14]:
fig2 = go.Figure()

categories = ["Small (< 200cm)", "Large (≥ 200cm)"]
rates = [8.18, 11.25]
counts = [4952, 5460]
bar_colors = ["#5DCAA5", "#E24B4A"]

for i, (cat, rate, n, color) in enumerate(zip(categories, rates, counts, bar_colors)):
    fig2.add_trace(go.Bar(
        x=[cat],
        y=[rate],
        marker_color=color,
        text=f"{rate}%<br>n={n}",
        textposition="outside",
        width=0.4,
        showlegend=False,
    ))

# Significance bracket
fig2.add_shape(type="line",
    x0=0, x1=1, y0=13.5, y1=13.5,
    line=dict(color="#444", width=1))
fig2.add_shape(type="line",
    x0=0, x1=0, y0=13.0, y1=13.5,
    line=dict(color="#444", width=1))
fig2.add_shape(type="line",
    x0=1, x1=1, y0=13.0, y1=13.5,
    line=dict(color="#444", width=1))
fig2.add_annotation(
    x=0.5, y=14.0,
    text="p<0.001  |  OR=1.42",
    showarrow=False,
    font=dict(size=11, color="#444")
)

fig2.update_layout(
    title=dict(
        text="H2 — Ghost net entanglement rate by animal size",
        font=dict(size=15)
    ),
    xaxis_title="Animal size class",
    yaxis=dict(
        title="Ghost net entanglement rate (%)",
        range=[0, 17],
        ticksuffix="%",
    ),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=135, l=60, r=40),
    width=500,
    height=420,
)

fig2.add_annotation(
    text="* Chi-square test. OR = odds ratio. Large defined as size_avg ≥ 200cm.",
    xref="paper", yref="paper",
    x=0, y=-0.15,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig2.show()
fig2.write_html("../figures/q3_h2_ghost_net_entanglement.html")

### H2 — Conclusion

**Hypothesis:** Ghost nets account for a disproportionate share of 
large animal entanglement.

**Result: Confirmed**

Chi-square test on the 2×2 contingency table (is_large × ghost_net_entanglement) 
shows a highly significant association (χ²=27.3, p<0.001).

| Size class       | Animals | Entanglement rate |
|------------------|---------|-------------------|
| Small (< 200cm)  | 4,952   | 8.18%             |
| Large (≥ 200cm)  | 5,460   | **11.25%**        |

Odds ratio = 1.42 — large animals are 42% more likely to be ghost net 
entangled than small animals.

**Limitation:** The effect is moderate, not absolute — ghost nets affect 
animals of all sizes. Size increases risk but is not the only factor.

### H3: Do different animal groups ingest different types of plastic?

In [15]:
# Check the plastic type columns
plastic_cols = ["hard", "soft", "rubber", "thread", "foam", "cloth", 
                "net", "rope", "line", "balloon", "bag", "bottle", 
                "fisheries", "nurdle"]

print("Non-zero counts per plastic type:")
print((species[plastic_cols] > 0).sum().sort_values(ascending=False))

print("\nWhich groups have the most ingestion records:")
print(species[species["has_ingestion"]==1]["group"].value_counts())

Non-zero counts per plastic type:
thread       1057
fisheries     940
hard          774
line          749
soft          560
rope          157
rubber        102
foam           85
bag            78
balloon        38
nurdle         24
net            16
cloth          15
bottle          2
dtype: int64

Which groups have the most ingestion records:
group
Manatee          646
Turtle           604
Shearwater       329
Prion            151
Toothed Whale     79
Albatross         17
Fulmar            16
Pinniped          15
Penguin            9
Baleen Whale       4
Tern               3
Kittiwake          3
Petrel             3
Wader              3
Gull               2
Booby              2
Waterfowl          1
Gannet             1
Name: count, dtype: int64


On plastic types — bottle (2) , cloth (15), net (16) and nurdle (24) are too rare to be meaningful. 
let's focus on the top ones: thread, fisheries, hard, line, soft, rope, rubber, foam, bag, balloon.

On animal groups — only 5 groups have enough ingestion records to compare reliably: 
Manatee (646), Turtle (604), Shearwater (329), Prion (151) and Toothed Whale (79). 
The rest are too small and would just add noise.

In [16]:
# Step 1 — Filter to the 5 main groups with enough ingestion records
main_groups = ["Manatee", "Turtle", "Shearwater", "Prion", "Toothed Whale"]
plastic_cols = ["thread", "fisheries", "hard", "line", "soft", 
                "rope", "rubber", "foam", "bag", "balloon"]

# Filter to animals that have ingestion AND are in our main groups
ingestion_df = species[
    (species["has_ingestion"] == 1) & 
    (species["group"].isin(main_groups))
].copy()

# Convert to presence/absence (1 if any plastic of that type found, 0 if not)
ingestion_df[plastic_cols] = (ingestion_df[plastic_cols] > 0).astype(int)

# For each group: what % of animals had each plastic type present?
profile = (
    ingestion_df.groupby("group")[plastic_cols]
    .mean() * 100
).round(1)

print(profile.to_string())

               thread  fisheries  hard  line  soft  rope  rubber  foam   bag  balloon
group                                                                                
Manatee          92.6       85.3   0.8  85.6   6.2   5.4     6.2   2.5   4.5      1.1
Prion             4.6        2.0  96.0   0.0   7.9   2.6     4.0   2.6   0.0      4.0
Shearwater        7.0        2.4  96.4   1.2   7.6   3.0     5.2   5.5   0.0      4.6
Toothed Whale    55.7       36.7  20.3  27.8  49.4  19.0     2.5   0.0  36.7      0.0
Turtle           59.3       54.6  42.7  25.7  69.9  14.7     4.1   7.3   2.2      0.8


Manatees — almost exclusively fishing-related debris: thread (92.6%), line (85.6%), fisheries (85.3%). 
They live in coastal/river areas near fishing activity, which makes perfect sense.

Prion & Shearwater — nearly identical profiles, dominated by hard plastic (96%). 
These are open ocean seabirds that mistake plastic fragments floating on the surface for food.

Toothed Whales — mixed profile, significant soft plastic (49.4%) and thread (55.7%). 
Opportunistic feeders that encounter many plastic types at depth.

Turtles — broad profile, highest soft plastic (69.9%), also thread and fisheries. 
Soft plastic like bags likely gets mistaken for jellyfish.

In [17]:
# Reorder columns by most interesting pattern
col_order = ["hard", "soft", "thread", "line", "fisheries", 
             "rope", "bag", "foam", "rubber", "balloon"]
group_order = ["Prion", "Shearwater", "Toothed Whale", "Turtle", "Manatee"]

# Reorder the profile dataframe
heatmap_data = profile[col_order].loc[group_order]

fig3 = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=[c.capitalize() for c in col_order],
    y=group_order,
    colorscale=[
        [0.0, "#E1F5EE"],
        [0.5, "#1D9E75"],
        [1.0, "#04342C"]
    ],
    text=heatmap_data.values,
    texttemplate="%{text}%",
    textfont=dict(size=11),
    hoverongaps=False,
    colorbar=dict(
        title="% of animals<br>with plastic type",
        ticksuffix="%"
    )
))

fig3.update_layout(
    title=dict(
        text="H3 — Plastic type ingestion profile by animal group",
        font=dict(size=15)
    ),
    xaxis=dict(title="Plastic type", side="bottom"),
    yaxis=dict(title="Animal group"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=135, l=120, r=40),
    width=750,
    height=400,
)

fig3.add_annotation(
    text="* % of animals with ingestion records where that plastic type was present. Groups with < 50 ingestion records excluded.",
    xref="paper", yref="paper",
    x=0, y=-0.18,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig3.show()
fig3.write_html("../figures/q3_h3_plastic_type_by_group.html")

### H3 — Conclusion

**Hypothesis:** Different animal groups ingest distinctly different 
types of plastic, reflecting their habitat and feeding behaviour.

**Result: Confirmed**

Presence/absence profiles across 5 groups (n ≥ 50 ingestion records) 
reveal clear ecological signatures:

| Group | Dominant plastic type | Likely explanation |
|---|---|---|
| Manatee | Thread, line, fisheries (85-93%) | Coastal habitat, entanglement in fishing debris |
| Turtle | Soft plastic, fisheries (55-70%) | Bags mistaken for jellyfish |
| Toothed Whale | Thread, soft, fisheries (37-56%) | Deep, opportunistic feeding |
| Shearwater | Hard plastic (96%) | Surface feeders mistaking fragments for prey |
| Prion | Hard plastic (96%) | Same as Shearwater — open ocean seabird |

**Key finding:** Seabirds (Prion, Shearwater) are almost exclusively 
affected by hard plastic fragments, while marine mammals and turtles 
show broader profiles dominated by fishing-related debris.

### H4: Microplastic contamination in commercially consumed fish species represents a measurable risk of human dietary exposure.

In [18]:
fish_to_human = pd.read_parquet(config["output_data"]["file10"])

In [19]:
# Sort by mp_per_individual
fish_to_human = fish_to_human.sort_values("mp_per_individual", ascending=True).copy()

# Colors by habitat
habitat_colors = {
    "Pelagic":  "#378ADD",
    "Demersal": "#E24B4A",
    "Farmed":   "#0A583A",
}

fig4 = go.Figure()

for _, row in fish_to_human.iterrows():
    color = habitat_colors[row["habitat"]]
    # Add bar
    fig4.add_trace(go.Bar(
        x=[row["mp_per_individual"]],
        y=[row["common_name"]],
        orientation="h",
        marker_color=color,
        marker_line_width=0,
        showlegend=False,
        text=f"{row['mp_per_individual']}",
        textposition="inside",
        textfont=dict(color="white", size=11),
        hovertemplate=(
            f"<b>{row['common_name']}</b><br>"
            f"MPs/individual: {row['mp_per_individual']}<br>"
            f"Habitat: {row['habitat']}<br>"
            f"Feeding: {row['feeding_type']}<br>"
            f"Region: {row['region']}<br>"
            f"Source: {row['source']}"
            "<extra></extra>"
        )
    ))

    # Star marker for species consumed whole
    if row["consumed_whole"]:
        fig4.add_annotation(
            x=row["mp_per_individual"] + 0.15,
            y=row["common_name"],
            text="★ eaten whole",
            showarrow=False,
            font=dict(size=10, color="#E24B4A"),
            xanchor="left",
            yanchor="middle",
        )

# Legend manually as annotations
fig4.add_annotation(
    x=7.5, y=1.5,
    text="■ Pelagic",
    showarrow=False,
    font=dict(size=11, color="#378ADD"),
    xanchor="left",
)
fig4.add_annotation(
    x=7.5, y=0.8,
    text="■ Demersal",
    showarrow=False,
    font=dict(size=11, color="#E24B4A"),
    xanchor="left",
)

fig4.add_annotation(
    x=7.5, y=0.1,
    text="■ Farmed",
    showarrow=False,
    font=dict(size=11, color="#0A583A"),
    xanchor="left",
)

# Average line
avg = fish_to_human["mp_per_individual"].mean()
fig4.add_vline(
    x=avg,
    line_dash="dash",
    line_color="#888",
    line_width=1,
    annotation_text=f"avg {avg:.1f}",
    annotation_position="top",
    annotation_font_size=10,
    annotation_font_color="#888",
)

fig4.update_layout(
    title=dict(
        text="Microplastics per individual in commercially consumed fish species",
        font=dict(size=15)
    ),
  xaxis=dict(
    title="Microplastic particles per individual fish (pieces/individual)",
    range=[0, 12],
    ),
    yaxis=dict(title=""),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=90, l=130, r=40),
    height=480,
    width=720,
    bargap=0.3,
)

fig4.add_annotation(
    text="* Data compiled from peer-reviewed studies (2020–2023).",
    xref="paper", yref="paper",
    x=0, y=-0.17,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig4.add_annotation(
    text="  MPs measured in gastrointestinal tract.",
    xref="paper", yref="paper",
    x=0, y=-0.23,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig4.add_annotation(
    text="  ★ = species typically consumed whole including gut.",
    xref="paper", yref="paper",
    x=0, y=-0.29,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig4.show()
fig4.write_html("../figures/q3_bonus_mp_in_edible_fish.html")

Pelagic fish live in the pelagic zone of ocean or lake waters—being neither close to the bottom nor near the shore;
in contrast with demersal fish that live on or near the bottom.

## H4 - Conclusion:

Microplastics in fish we eat

Values represent mean microplastic **particles** (pieces) found in the 
gastrointestinal tract per individual fish across 12 commercially consumed 
species (data compiled from peer-reviewed studies 2020–2023).

**Key findings:**
- Demersal species (bottom dwellers) carry significantly more MPs than 
  pelagic species - up to 9 pieces in Gilthead seabream
- Farmed species have the most microplastics in them — up to 9.3 pieces/individual in rainbow trout
- Sardines and anchovies have lower counts (1.4–2.3) but are consumed 
  whole including the gut — so all particles are ingested directly
- The global average across fish species is ~5 MPs/individual
- Fibers are the dominant particle type across all species

**What this means:** An average seafood consumer eating fish several 
times per week could be ingesting thousands of microplastic particles 
per year through fish consumption alone.

## Machine learning model:
Can we predict which marine animals are at risk of plastic ingestion based on their species, size, location and local microplastic concentration?

In [20]:
# Add concentration_class back from the spatial join results
species_ml = species_clean.copy()

# concentration_class is already on species_clean from step 3
# just check it's there
print(species_ml.columns.tolist())
print(species_ml["concentration_class"].value_counts(dropna=False))

['citation', 'id', 'taxa', 'group', 'species', 'family', 'age', 'size', 'size_avg', 'sex', 'latitude', 'longitude', 'death', 'cod', 'harddeath', 'softdeath', 'threaddeath', 'rubberdeath', 'foamdeath', 'clothdeath', 'nrdeath', 'hard', 'soft', 'rubber', 'thread', 'foam', 'cloth', 'net', 'rope', 'line', 'balloon', 'bag', 'bottle', 'fisheries', 'nurdle', 'nr', 'total', 'volume', 'mass', 'acute_injury', 'cod_label', 'is_entangled', 'has_ingestion', 'ghost_net_entanglement', 'log_size', 'is_large', 'gyre_region', 'dist_to_nearest_mp_km', 'within_radius', 'concentration_class']
concentration_class
NaN          3553
Medium       2954
Very Low     2707
Low           588
High          230
Very High     173
Name: count, dtype: int64


In [21]:
feature_cols = [
    "group", "is_large", "latitude", "longitude",
    "gyre_region", "concentration_class",
    "hard", "soft", "thread", "line",
    "fisheries", "rope", "foam", "bag"
]

target = "has_ingestion"

# Fill NaN concentration_class with "Unknown"
species_ml["concentration_class"] = (
    species_ml["concentration_class"]
    .astype(str)
    .replace("nan", "Unknown")
)

# Check class balance
print("=== Target distribution ===")
print(species_ml[target].value_counts())
print(f"Positive rate: {species_ml[target].mean():.1%}")

# Check rows available after dropna
subset = species_ml[feature_cols + [target]].dropna()
print(f"\n=== Rows with all features complete ===")
print(f"Full dataset : {len(species_ml):,}")
print(f"After dropna : {len(subset):,}")
print(f"Dropped      : {len(species_ml) - len(subset):,}")

# Missing per column
print("\n=== Missing values per feature ===")
print(species_ml[feature_cols].isnull().sum())

=== Target distribution ===
has_ingestion
0    8416
1    1789
Name: count, dtype: int64
Positive rate: 17.5%

=== Rows with all features complete ===
Full dataset : 10,205
After dropna : 10,205
Dropped      : 0

=== Missing values per feature ===
group                  0
is_large               0
latitude               0
longitude              0
gyre_region            0
concentration_class    0
hard                   0
soft                   0
thread                 0
line                   0
fisheries              0
rope                   0
foam                   0
bag                    0
dtype: int64


One thing to note: 17.5% positive rate means the classes are imbalanced — about 5 non-ingestion cases for every 1 ingestion case. 
Not severe enough to panic, but we need to tell the model about it so it doesn't just predict "no ingestion" for everything and get 82% accuracy while being useless.

preprocessing and train/test split

In [22]:
# Encode categorical columns
species_ml_enc = species_ml[feature_cols + [target]].copy()

cat_cols = ["group", "gyre_region", "concentration_class"]
encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    species_ml_enc[col] = le.fit_transform(species_ml_enc[col].astype(str))
    encoders[col] = le  # save for later inspection

# Split
X = species_ml_enc[feature_cols]
y = species_ml_enc[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,}")
print(f"Test size  : {len(X_test):,}")
print(f"\nPositive rate in train : {y_train.mean():.1%}")
print(f"Positive rate in test  : {y_test.mean():.1%}")

Train size : 8,164
Test size  : 2,041

Positive rate in train : 17.5%
Positive rate in test  : 17.5%


Train a Random Forest
Random Forrest is the right first choice here because it handles mixed feature types well 
(we have categorical, binary and continuous features all mixed together), 
it's robust to class imbalance with class_weight="balanced", 
it gives us feature importance for free which makes a great portfolio visualization, 
and it rarely needs much tuning to get a solid baseline.

In [23]:
# Train
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print("=== Classification report ===")
print(classification_report(y_test, y_pred, 
      target_names=["No ingestion", "Ingestion"]))

print(f"ROC-AUC score : {roc_auc_score(y_test, y_prob):.3f}")

print("\n=== Confusion matrix ===")
cm = confusion_matrix(y_test, y_pred)
print(cm)

=== Classification report ===
              precision    recall  f1-score   support

No ingestion       0.99      1.00      1.00      1683
   Ingestion       0.99      0.97      0.98       358

    accuracy                           0.99      2041
   macro avg       0.99      0.98      0.99      2041
weighted avg       0.99      0.99      0.99      2041

ROC-AUC score : 0.989

=== Confusion matrix ===
[[1680    3]
 [  11  347]]


That's an almost perfect model — but it's too good, which is actually a red flag that needs to investigate before celebrating.
ROC-AUC of 0.989, 99% accuracy, only 14 mistakes out of 2,041. Real-world biological prediction problems don't perform like this. Something is likely causing data leakage — meaning some of our features are directly encoding the answer.

In [24]:
# Check correlation between plastic type features and target
print("=== Feature correlations with has_ingestion ===")
print(species_ml_enc[feature_cols + [target]]
      .corr()[target]
      .sort_values(ascending=False)
      .round(3))

=== Feature correlations with has_ingestion ===
has_ingestion          1.000
hard                   0.217
thread                 0.183
group                  0.179
soft                   0.177
fisheries              0.169
line                   0.143
rope                   0.133
longitude              0.133
foam                   0.123
bag                    0.079
gyre_region            0.050
concentration_class   -0.073
is_large              -0.120
latitude              -0.254
Name: has_ingestion, dtype: float64


The correlations are moderate — nothing suspiciously close to 1.0 — so it's not a straightforward leakage case. 
But logically:
The plastic type columns (hard, soft, thread etc.) are still a problem. They represent plastic found inside the animal. 
If an animal has hard > 0 it almost certainly has_ingestion = 1 by definition — they're measuring the same thing from different angles. 
The model is essentially learning "if plastic found → ingestion = true" which is circular.

In [25]:
# If any plastic type > 0, does has_ingestion always = 1?
plastic_cols = ["hard", "soft", "thread", "line", 
                "fisheries", "rope", "foam", "bag"]

has_any_plastic = (species_ml[plastic_cols] > 0).any(axis=1)

print("Animals with any plastic type > 0:")
print(pd.crosstab(has_any_plastic, species_ml["has_ingestion"]))

Animals with any plastic type > 0:
has_ingestion     0     1
row_0                    
False          8415    31
True              1  1758


confirmed leakage. 
If any plastic type column is > 0, the animal has ingestion in 1,758 out of 1,759 cases. 
The model wasn't learning anything meaningful, it was just reading the answer directly from the data.
The 31 animals with has_ingestion = 1 but no plastic type columns — and the 1 animal with plastic but no ingestion flag — 
are just minor data entry inconsistencies in the original dataset.

So we rebuild with only the honest features

In [26]:
# Honest feature set — only what we'd know BEFORE examining the animal
honest_features = [
    "group", "is_large", "latitude", "longitude",
    "gyre_region", "concentration_class"
]

X_honest = species_ml_enc[honest_features]
y_honest = species_ml_enc[target]

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_honest, y_honest, 
    test_size=0.2, random_state=42, stratify=y_honest
)

rf2 = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf2.fit(X_train2, y_train2)

y_pred2 = rf2.predict(X_test2)
y_prob2 = rf2.predict_proba(X_test2)[:, 1]

print("=== Classification report ===")
print(classification_report(y_test2, y_pred2,
      target_names=["No ingestion", "Ingestion"]))

print(f"ROC-AUC score : {roc_auc_score(y_test2, y_prob2):.3f}")

print("\n=== Confusion matrix ===")
print(confusion_matrix(y_test2, y_pred2))

=== Classification report ===
              precision    recall  f1-score   support

No ingestion       0.89      0.88      0.89      1683
   Ingestion       0.47      0.51      0.49       358

    accuracy                           0.81      2041
   macro avg       0.68      0.69      0.69      2041
weighted avg       0.82      0.81      0.82      2041

ROC-AUC score : 0.776

=== Confusion matrix ===
[[1481  202]
 [ 177  181]]


ROC-AUC 0.776 — meaningfully better than random (0.5) and better than a naive "always predict no ingestion" baseline. The model is genuinely learning something.
No ingestion: 89% precision, 88% recall — the model is good at identifying animals that won't ingest plastic.
Ingestion: 47% precision, 51% recall — weaker, which makes biological sense. Plastic ingestion is hard to predict from location and species alone — there's a lot of individual variation.
The confusion matrix:

1,481 correctly identified as no ingestion ✓
181 correctly identified as ingestion ✓
202 false alarms (predicted ingestion, actually clean)
177 missed ingestion cases

After removing data leakage, the model achieves ROC-AUC 0.776, suggesting that species group, size, location and 
local microplastic concentration have moderate predictive power for plastic ingestion risk — but individual variation remains high.

Let's check feature importance to see what the model actually learned:

In [27]:
# Check correlations between features
# For encoded numerics only
corr = species_ml_enc[honest_features].corr().round(2)
print(corr)

                     group  is_large  latitude  longitude  gyre_region  \
group                 1.00     -0.59     -0.07       0.21         0.04   
is_large             -0.59      1.00      0.28      -0.59        -0.09   
latitude             -0.07      0.28      1.00      -0.52        -0.12   
longitude             0.21     -0.59     -0.52       1.00        -0.01   
gyre_region           0.04     -0.09     -0.12      -0.01         1.00   
concentration_class  -0.27      0.26      0.08      -0.17        -0.01   

                     concentration_class  
group                              -0.27  
is_large                            0.26  
latitude                            0.08  
longitude                          -0.17  
gyre_region                        -0.01  
concentration_class                 1.00  


In [28]:
corr = species_ml_enc[honest_features].corr().round(2)

# Nicer display names
labels = [
    "Group", "Is large", "Latitude", 
    "Longitude", "Gyre region", "Conc. class"
]

# Upper triangle mask — keep only lower triangle
mask = np.tril(np.ones_like(corr, dtype=bool))
corr_masked = corr.where(mask)

fig_corr = go.Figure(go.Heatmap(
    z=corr_masked.values,
    x=labels,
    y=labels,
    colorscale=[
        [0.0,  "#042C53"],  # deep ocean blue — strong negative
        [0.25, "#185FA5"],
        [0.5,  "#E1F5EE"],  # pale seafoam — zero correlation
        [0.75, "#1D9E75"],
        [1.0,  "#04342C"],  # deep teal — strong positive
    ],
    zmin=-1, zmax=1,
    text=corr_masked.round(2).values,
    texttemplate="%{text}",
    textfont=dict(size=12),
    hoverongaps=False,
    colorbar=dict(
        title="Correlation",
        tickvals=[-1, -0.5, 0, 0.5, 1],
        ticktext=["-1", "-0.5", "0", "0.5", "1"],
    )
))

fig_corr.update_layout(
    title=dict(
        text="Feature correlation matrix",
        font=dict(size=15)
    ),
    xaxis=dict(side="bottom"),
    yaxis=dict(autorange="reversed"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=80, l=120, r=40),
    width=580,
    height=520,
)

fig_corr.update_traces(
    texttemplate="%{text}",
    text=[
        [str(v) if not np.isnan(v) else "" 
         for v in row]
        for row in corr_masked.values
    ]
)

fig_corr.show()
fig_corr.write_html("../figures/ml_feature_correlation.html")

Two moderate correlations worth noting:

group vs is_large (-0.59) — makes sense, certain groups are consistently large or small
longitude vs is_large (-0.59) and latitude vs longitude (-0.52) — geographic patterns in where large animals are studied

These are acceptable. The rule of thumb is worry above 0.8, flag between 0.6-0.8, relax below 0.6.

In [29]:
importances = pd.Series(
    rf2.feature_importances_, 
    index=honest_features
).sort_values(ascending=True)

print(importances)

gyre_region            0.001684
is_large               0.019413
concentration_class    0.027180
group                  0.151858
longitude              0.390734
latitude               0.409132
dtype: float64


Location dominates — latitude (40.9%) and longitude (39.1%) together account for 80% of the predictive power. 
Where an animal is found matters far more than what it is.
Species group matters moderately — 15.2%, which makes sense given the distinct ingestion profiles we saw in H3.
Concentration class, size and gyre region are weak predictors — together only 4.8%. 
Surprising for concentration class given what we found in H1, but the spatial coordinates are probably capturing the same information more precisely.

In [30]:
fig_imp = go.Figure(go.Bar(
    x=importances.values,
    y=importances.index,
    orientation="h",
    marker_color=[
        "#378ADD" if f in ["latitude", "longitude"] 
        else "#1D9E75" if f == "group"
        else "#888780" 
        for f in importances.index
    ],
    text=[f"{v:.1%}" for v in importances.values],
    textposition="outside",
))

fig_imp.update_layout(
    title=dict(
        text="Feature importance — predicting plastic ingestion risk",
        font=dict(size=15)
    ),
    xaxis=dict(
        title="Feature importance",
        tickformat=".0%",
        range=[0, 0.55]
    ),
    yaxis=dict(title=""),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=80, l=150, r=60),
    height=380,
    width=680,
)

fig_imp.show()
fig_imp.write_html("../figures/ml_feature_importance.html")

# ROC curve
fpr, tpr, _ = roc_curve(y_test2, y_prob2)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode="lines",
    line=dict(color="#378ADD", width=2),
    name=f"Random Forest (AUC = 0.776)",
))
fig_roc.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    line=dict(color="#888", dash="dash", width=1),
    name="Random baseline (AUC = 0.5)",
))

fig_roc.update_layout(
    title=dict(
        text="ROC curve — plastic ingestion risk model",
        font=dict(size=15)
    ),
    xaxis=dict(title="False positive rate"),
    yaxis=dict(title="True positive rate"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=60, l=60, r=40),
    height=420,
    width=560,
    legend=dict(x=0.4, y=0.1),
)

fig_roc.show()
fig_roc.write_html("../figures/ml_roc_curve.html")

Now step 2 — hyperparameter tuning with RandomizedSearchCV. I'll explain what we're tuning and why before you run it:

n_estimators — number of trees. More is usually better but slower
max_depth — how deep each tree grows. Too deep = overfitting
min_samples_split — minimum samples needed to split a node. Higher = simpler trees
max_features — how many features each tree considers at each split

We use RandomizedSearchCV rather than GridSearchCV because it samples random combinations instead of trying every single one — much faster with similar results:

In [31]:
param_dist = {
    "n_estimators":      randint(100, 500),
    "max_depth":         [None, 5, 10, 15, 20],
    "min_samples_split": randint(2, 20),
    "max_features":      ["sqrt", "log2", 0.5],
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    param_distributions=param_dist,
    n_iter=30,          # try 30 random combinations
    scoring="roc_auc",  # optimise for AUC not accuracy
    cv=5,               # 5-fold cross validation
    random_state=42,
    verbose=1,
    n_jobs=-1
)

rf_search.fit(X_train2, y_train2)

print(f"Best parameters : {rf_search.best_params_}")
print(f"Best CV AUC     : {rf_search.best_score_:.3f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters : {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 18, 'n_estimators': 149}
Best CV AUC     : 0.814


CV AUC went from 0.776 to 0.814 just from tuning. Let's now evaluate the tuned model on the test set and then try Gradient Boosting to see if we can push it further

In [32]:
# Evaluate tuned Random Forest on test set
rf_tuned = rf_search.best_estimator_

y_pred_tuned = rf_tuned.predict(X_test2)
y_prob_tuned = rf_tuned.predict_proba(X_test2)[:, 1]

print("=== Tuned Random Forest ===")
print(classification_report(y_test2, y_pred_tuned,
      target_names=["No ingestion", "Ingestion"]))
print(f"ROC-AUC : {roc_auc_score(y_test2, y_prob_tuned):.3f}")

=== Tuned Random Forest ===
              precision    recall  f1-score   support

No ingestion       0.91      0.84      0.87      1683
   Ingestion       0.44      0.59      0.51       358

    accuracy                           0.80      2041
   macro avg       0.68      0.72      0.69      2041
weighted avg       0.82      0.80      0.81      2041

ROC-AUC : 0.804


ROC-AUC improved from 0.776 to 0.804. Notice the tradeoff though:

- Ingestion recall went up from 51% to 59% — the model catches more actual ingestion cases
- Ingestion precision went down from 47% to 44% — more false alarms
But, that's the right direction for this problem — missing a real ingestion case is worse than a false alarm

In [33]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
)

# Handle class imbalance manually via sample weights
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train2)

gb.fit(X_train2, y_train2, sample_weight=sample_weights)

y_pred_gb = gb.predict(X_test2)
y_prob_gb = gb.predict_proba(X_test2)[:, 1]

print("=== Gradient Boosting ===")
print(classification_report(y_test2, y_pred_gb,
      target_names=["No ingestion", "Ingestion"]))
print(f"ROC-AUC : {roc_auc_score(y_test2, y_prob_gb):.3f}")

=== Gradient Boosting ===
              precision    recall  f1-score   support

No ingestion       0.91      0.81      0.86      1683
   Ingestion       0.41      0.62      0.49       358

    accuracy                           0.78      2041
   macro avg       0.66      0.71      0.67      2041
weighted avg       0.82      0.78      0.79      2041

ROC-AUC : 0.801


In [34]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric="auc",
    verbosity=0,
)

sample_weights = compute_sample_weight("balanced", y_train2)
xgb_model.fit(X_train2, y_train2, sample_weight=sample_weights)

y_pred_xgb = xgb_model.predict(X_test2)
y_prob_xgb = xgb_model.predict_proba(X_test2)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test2, y_pred_xgb,
      target_names=["No ingestion", "Ingestion"]))
print(f"ROC-AUC : {roc_auc_score(y_test2, y_prob_xgb):.3f}")

=== XGBoost ===
              precision    recall  f1-score   support

No ingestion       0.91      0.76      0.83      1683
   Ingestion       0.37      0.64      0.47       358

    accuracy                           0.74      2041
   macro avg       0.64      0.70      0.65      2041
weighted avg       0.81      0.74      0.77      2041

ROC-AUC : 0.792


In [ ]:
# Separate predicted probabilities by actual class
prob_no_ingestion = y_prob_tuned[y_test2 == 0]
prob_ingestion = y_prob_tuned[y_test2 == 1]

fig_prob = go.Figure()

fig_prob.add_trace(go.Histogram(
    x=prob_no_ingestion,
    name="No ingestion (actual)",
    opacity=0.7,
    marker_color="#5DCAA5",
    nbinsx=40,
))

fig_prob.add_trace(go.Histogram(
    x=prob_ingestion,
    name="Ingestion (actual)",
    opacity=0.7,
    marker_color="#E24B4A",
    nbinsx=40,
))

fig_prob.add_vline(
    x=0.5,
    line_dash="dash",
    line_color="#444",
    line_width=1,
    annotation_text="decision threshold 0.5",
    annotation_position="top right",
    annotation_font_size=10,
)

fig_prob.update_layout(
    barmode="overlay",
    title=dict(
        text="Predicted ingestion probability by actual class",
        font=dict(size=15)
    ),
    xaxis=dict(title="Predicted probability of ingestion"),
    yaxis=dict(title="Number of animals"),
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="sans-serif", size=12),
    margin=dict(t=80, b=60, l=60, r=40),
    width=700,
    height=420,
    legend=dict(x=0.55, y=0.95),
)

fig_prob.add_annotation(
    text="* Tuned Random Forest. Good separation = two distinct peaks on opposite sides of the threshold.",
    xref="paper", yref="paper",
    x=0, y=-0.12,
    showarrow=False,
    font=dict(size=9, color="#888"),
    xanchor="left"
)

fig_prob.show()
fig_prob.write_html("../figures/ml_probability_distribution.html")

The green mass (no ingestion) peaks sharply near 0 — the model is very confident about animals that don't ingest plastic. Good.
The red (ingestion) is spread more broadly across 0.2 to 1.0 — the model is less certain about ingestion cases, which explains the lower recall.
The overlap in the 0.2–0.5 range is where most mistakes happen — animals the model is uncertain about. 
This is the "hard zone" where biology alone can't clearly predict ingestion.

The honest interpretation:

The model confidently identifies non-ingestion cases but struggles with certainty on ingestion cases 
— reflecting genuine biological variability rather than a model weakness. Some animals ingest plastic regardless of location or species group.

### Model comparison — summary

The model is good at ranking animals by ingestion risk (AUC 0.804), 
but not confident enough to make a definitive yes/no call for individual animals — particularly for ingestion cases which show high uncertainty.

| Model | ROC-AUC | Ingestion recall |
|---|---|---|
| Baseline Random Forest | 0.776 | 51% |
| Tuned Random Forest | 0.804 | 59% |
| Gradient Boosting | 0.801 | 62% |
| XGBoost | 0.792 | 64% |

**Selected model: Tuned Random Forest (ROC-AUC 0.804)**

All models perform similarly, suggesting the predictive ceiling 
is set by the available features rather than model choice. 
Location (latitude/longitude) dominates all models — 
where an animal lives is the strongest predictor of 
plastic ingestion risk.

Gradient Boosting and XGBoost trade precision for recall — 
useful if the priority is catching every at-risk animal, 
at the cost of more false alarms.

**What is ROC-AUC?**

**ROC** = Receiver Operating Characteristic
**AUC** = Area Under the Curve

**The intuition**
Our model scores every animal between 0 and 1 — the predicted 
probability that it has ingested plastic. We need to pick a 
threshold to make a decision:

- **Low threshold (e.g. 0.2)** → catches almost every ingestion 
  case but also flags many healthy animals. High recall, low precision.
- **High threshold (e.g. 0.8)** → only flags cases the model is 
  very confident about, but misses many real cases. High precision, 
  low recall.

The ROC curve plots **every possible threshold at once**:
- Y-axis: true positive rate (how many real ingestion cases we catch)
- X-axis: false positive rate (how many healthy animals we wrongly flag)

**What AUC means**
| AUC | Interpretation |
|-----|---------------|
| 0.5 | No better than random guessing |
| 0.7 | Acceptable |
| 0.8 | Good |
| 0.9 | Excellent |
| 1.0 | Perfect (usually means data leakage!) |

**The simplest way to think about it**
> If you pick one animal that DID ingest plastic and one that DIDN'T,
> AUC is the probability that the model correctly scores the ingestion
> animal higher.

Our tuned Random Forest scores **0.804** — meaning there is an 80.4% 
chance the model correctly ranks an ingestion case above a 
non-ingestion case. Meaningfully better than random (50%), 
achieved using only 6 features known before examining the animal.